# 02 · Strings and f-strings

After this notebook you can slice, search, clean and format text fluently, explain why `len("😀")` is 1 while its UTF-8 encoding is 4 bytes, and build long strings efficiently — the everyday plumbing of prompts, documents and model output.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · Variables, types and numbers](01_variables_types_numbers.ipynb)

## Why this matters in interviews

- Every LLM application is string plumbing: assembling prompts, cleaning documents before chunking and embedding, parsing what the model sends back.
- Warm-up coding questions are often string questions — reverse, palindrome, count characters, parse a line — and slicing plus the core methods turn them into one-liners.
- Unicode is where confident candidates slip: characters vs bytes vs tokens, `lower()` vs `casefold()`, two strings that look identical but compare unequal.
- A classic performance follow-up: why `"".join(parts)` rather than `+=` in a loop.

## What interviewers ask

- Reverse a string; check whether it is a palindrome, ignoring case and punctuation.
- Why are strings immutable? What happens on `s[0] = "x"`?
- `split()` vs `split(" ")`? `find` vs `index`? What does `strip("abc")` actually remove?
- Format a number with 2 decimals, a thousands separator, as a percentage, or right-aligned in a column.
- What is the difference between `str` and `bytes`? What is UTF-8? Why can `len(text)` differ from its size in bytes?
- Why is `s += piece` in a loop discouraged, and what do you use instead?
- How do you compare two strings case-insensitively — properly?

## 1 · Indexing and slicing

A string is an immutable **sequence** of characters (Unicode code points). Index from the front with `0, 1, 2 …` or from the back with `-1, -2 …`. A slice `s[start:stop:step]` includes `start`, **excludes** `stop`, and any part can be left out.

```
 +---+---+---+---+---+---+
 | P | y | t | h | o | n |
 +---+---+---+---+---+---+
   0   1   2   3   4   5     positive index
  -6  -5  -4  -3  -2  -1     negative index
```

**Predict, then run:** `s[1:4]`, `s[-3:]`, `s[::2]`, `s[::-1]`, `s[4:1:-1]`, `s[10:]`.

In [ ]:
s = "Python"
examples = [("s[0]", s[0]), ("s[-1]", s[-1]), ("s[1:4]", s[1:4]), ("s[:2]", s[:2]), ("s[-3:]", s[-3:]),
            ("s[::2]", s[::2]), ("s[::-1]", s[::-1]), ("s[4:1:-1]", s[4:1:-1]), ("s[10:]", s[10:])]
for expr, value in examples:
    print(f"{expr:<10} -> {value!r}")
try:
    s[10]
except IndexError as err:
    print("s[10]      -> IndexError:", err)

assert s[::-1] == "nohtyP" and s[4:1:-1] == "oht" and s[10:] == "" and len(s[1:4]) == 4 - 1

A slice never raises for out-of-range bounds — it just returns less — which makes it the safe way to **truncate**. Everyday AI uses: a preview of the first N characters of a document, the *tail* of a long log or conversation, or a cut at the last space so no word is split.

In [ ]:
doc = "Refunds are processed within 5 business days. Contact support for exceptions."
print("preview :", repr(doc[:30]))
print("tail    :", repr(doc[-22:]))
cut = doc[:30].rsplit(" ", 1)[0] + "…"           # drop the partial last word
print("word cut:", repr(cut))
print("'short'[:500] is safe:", repr("short"[:500]))
assert "short"[:500] == "short" and cut == "Refunds are processed within…"

## 2 · Strings are immutable

You cannot change a string in place. Every "modifying" method — `upper`, `replace`, `strip` — returns a **new** string and leaves the original alone, so you must use the return value. Immutability is what lets strings be dict keys, be shared safely between names, and be cached (interned) by CPython.

In [ ]:
greeting = "hello"
try:
    greeting[0] = "H"
except TypeError as err:
    print("TypeError:", err)

greeting.upper()                           # result thrown away — a common bug
print("after greeting.upper():", greeting)
greeting = greeting.upper()                # rebind the name to the new string
print("after rebinding       :", greeting)
print("'change' a character  :", "J" + "hello"[1:])
assert greeting == "HELLO"

## 3 · The core methods

| Method | What it does | Watch out |
|---|---|---|
| `split(sep=None, maxsplit=-1)`, `rsplit`, `splitlines()` | break into a list | `split()` with no argument splits on **runs** of whitespace and drops empties; `split(" ")` does neither |
| `sep.join(items)` | glue an iterable of strings | the items must already be `str` |
| `strip` / `lstrip` / `rstrip` | remove characters from the ends | the argument is a **set of characters**, not a prefix |
| `removeprefix` / `removesuffix` (3.9+) | remove one exact prefix / suffix | the right tool for "strip this marker" |
| `replace(old, new, count=-1)` | replace all (or `count`) occurrences | returns a new string |
| `find` / `rfind` → index or `-1`; `index` → index or `ValueError` | search | `-1` is also a valid *slice* index — check for it |
| `startswith` / `endswith` | prefix / suffix test | accept a **tuple** of options |
| `lower`, `upper`, `title`, `casefold` | change case | use `casefold()` for comparisons |
| `partition(sep)` / `rpartition` | split **once** into `(before, sep, after)` | always a 3-tuple, even when `sep` is missing |
| `count(sub)`, `sub in s` | count / membership | `count` skips overlapping matches |

**Predict, then run:** `"  many   spaces ".split()` vs `.split(" ")`.

In [ ]:
messy = "  many   spaces "
print("split()     :", messy.split())
print("split(' ')  :", messy.split(" "))
print("csv split   :", "a,b,,c".split(","), "| maxsplit=1:", "k=v=w".split("=", 1))
print("join        :", " | ".join(["system", "user", "assistant"]))
print("splitlines  :", "line 1\nline 2\r\nline 3".splitlines())
print("replace     :", "gpt-4 is great; gpt-4 is fast".replace("gpt-4", "the model", 1))
print("find        :", "tokenizer".find("ize"), "tokenizer".find("xyz"))
try:
    "tokenizer".index("xyz")
except ValueError as err:
    print("index       : ValueError:", err)
print("endswith    :", "notes.md".endswith((".md", ".txt")), "| startswith:", "/help".startswith("/"))
assert messy.split() == ["many", "spaces"] and messy.split(" ") == ["", "", "many", "", "", "spaces", ""]

### The `strip` trap — and its LLM-output version

`lstrip("Answer: ")` does not remove the prefix `"Answer: "`. It removes **any** of the characters `A n s w e r : space` from the left, as many as it finds, in any order — so it keeps eating into the answer. Models often wrap output in labels or Markdown fences, so this bug is common in parsing code. Use `removeprefix` / `removesuffix`.

**Predict, then run:** what is left of `"Answer: seven"`?

In [ ]:
import json

reply = "Answer: seven"
print("lstrip('Answer: ')      :", repr(reply.lstrip("Answer: ")))
print("removeprefix('Answer: '):", repr(reply.removeprefix("Answer: ")))
print("strip('w.moc')          :", repr("www.example.com".strip("w.moc")))

fenced = '```json\n{"answer": 7}\n```'
payload = fenced.removeprefix("```json").removesuffix("```").strip()
print("fence removed           :", repr(payload), "->", json.loads(payload))
assert reply.lstrip("Answer: ") == "ven" and reply.removeprefix("Answer: ") == "seven" and json.loads(payload) == {"answer": 7}

### Case: `lower`, `upper`, `title`, `casefold`

For case-insensitive **comparison**, use `casefold()` — `lower()` made aggressive for Unicode (German `ß` becomes `ss`). `title()` capitalises after every non-letter, apostrophes included.

In [ ]:
print("title()   :", "they're bill's friends".title())
print("lower()   :", "Straße".lower(), "| casefold():", "Straße".casefold())
print("'STRASSE' vs 'Straße' — equal after lower()?", "STRASSE".lower() == "Straße".lower(),
      "| after casefold()?", "STRASSE".casefold() == "Straße".casefold())
assert "STRASSE".casefold() == "Straße".casefold() and "STRASSE".lower() != "Straße".lower()

### `partition` for parsing model output

`partition(sep)` splits **once** and always returns a 3-tuple, so it never raises — ideal for `key: value` lines such as the `Thought / Action` lines a ReAct-style agent prints. `in` and `count` answer "whether" and "how often".

In [ ]:
agent_output = """Thought: I should look this up.
Action: search[python generators]
Observation: pending"""

fields = {}
for line in agent_output.splitlines():
    key, sep, value = line.partition(": ")
    if sep:                                     # skip lines without "key: "
        fields[key] = value
print(fields)

tool, _, arg = fields["Action"].partition("[")
print("tool:", tool, "| argument:", arg.removesuffix("]"))
print("'search' in output:", "search" in agent_output, "| 'aaaa'.count('aa'):", "aaaa".count("aa"), "(non-overlapping)")
assert (tool, arg.removesuffix("]")) == ("search", "python generators") and "aaaa".count("aa") == 2

## 4 · f-strings and the format mini-language

An f-string evaluates each `{expression}` and formats it with an optional spec after a colon:
`{value!conversion:[fill][align][sign][width][,][.precision][type]}`. The specs you actually use:

| Spec | Example | Result |
|---|---|---|
| `.2f` | `f"{0.87654:.2f}"` | `0.88` |
| `,` or `_` | `f"{1234567:,}"` | `1,234,567` |
| `.1%` | `f"{0.4213:.1%}"` | `42.1%` |
| `>w`, `<w`, `^w` | `f"{'ok':>6}"` | `    ok` |
| fill + align | `f"{'mid':*^11}"` | `****mid****` |
| `05d`, `#x`, `08b`, `.2e` | `f"{42:05d}"` | `00042` |
| `!r` | `f"{'hi'!r}"` | `'hi'` — the `repr`, quotes shown |
| `=` (3.8+, debugging) | `f"{score=:.2f}"` | `score=0.88` |

In [ ]:
score, tokens, ratio, model = 0.87654, 1_234_567, 0.4213, "gpt-4.1-mini"
width = 16
formatted = {
    "score:.2f": f"{score:.2f}",
    "tokens:,": f"{tokens:,}",
    "tokens:_": f"{tokens:_}",
    "ratio:.1%": f"{ratio:.1%}",
    "model:>16": f"{model:>16}|",
    "model:<16": f"{model:<16}|",
    "model:^16": f"{model:^16}|",
    "'mid':*^11": f"{'mid':*^11}",
    "42:05d": f"{42:05d}",
    "255:#x, 5:08b": f"{255:#x}, {5:08b}",
    "0.000123:.2e": f"{0.000123:.2e}",
    "model!r": f"{model!r}",
    "score=": f"{score=}",
    "score=:.2f": f"{score=:.2f}",
    "model:>{width}": f"{model:>{width}}|",
    "{{ }} literal": f"{{braces}} {tokens // 1000}k",
}
for spec, out in formatted.items():
    print(f"{spec:<15} -> {out}")
assert formatted["score:.2f"] == "0.88" and formatted["tokens:,"] == "1,234,567" and formatted["ratio:.1%"] == "42.1%"
assert formatted["score=:.2f"] == "score=0.88" and formatted["model!r"] == "'gpt-4.1-mini'"

Width and alignment turn raw results into a readable report — say, a small model comparison (the numbers are made up):

In [ ]:
results = [("gpt-4.1-mini", 812, 0.8123), ("qwen2.5:3b", 145, 0.7011), ("llama-3.1-8b", 1260, 0.7555)]
print(f"{'model':<14}{'p50 ms':>8}{'accuracy':>10}")
print("-" * 32)
for name, latency_ms, acc in sorted(results, key=lambda r: -r[2]):
    print(f"{name:<14}{latency_ms:>8,}{acc:>10.1%}")

### `str.format`, `%`, and prompt templates that contain JSON

- `str.format` speaks the same mini-language, but its template can be **data** — loaded from a file or config — which an f-string cannot (an f-string is evaluated where it is written).
- `%`-formatting is the old style; you still meet it in `logging` calls (`log.info("took %s ms", ms)`), where formatting is deferred until the message is actually emitted.
- **The prompt-template trap:** a template containing a JSON example has literal `{` and `}`, which `str.format` treats as fields — `KeyError`. Double the braces, or use `string.Template` (`$name` placeholders), which leaves braces alone. And never call `.format()` on a template a user supplied: fields can reach attributes (`{0.__class__}`).

In [ ]:
from string import Template

print("{} has {:,} tokens".format("doc-7", 5120), "|", "{name}: {score:.1%}".format(name="recall@5", score=0.8))
print("%s took %.1f ms" % ("retrieval", 42.25))       # 42.2: formatting also rounds half to even (notebook 01)

prompt_template = 'Reply as JSON like {"answer": "..."}. Question: {question}'
try:
    prompt_template.format(question="What is RAG?")
except KeyError as err:
    print("KeyError:", err)

escaped = 'Reply as JSON like {{"answer": "..."}}. Question: {question}'
print(escaped.format(question="What is RAG?"))
tmpl = Template('Reply as JSON like {"answer": "..."}. Question: $question')
print(tmpl.substitute(question="What is RAG?"))
assert escaped.format(question="Q") == tmpl.substitute(question="Q")

## 5 · Escapes, raw strings and multi-line prompts

- Backslash escapes: `\n` newline, `\t` tab, `\\` backslash, `\u00e9` a Unicode character. So the Windows path `"C:\new\table"` silently contains a newline and a tab.
- A **raw string** `r"..."` keeps backslashes literally — use it for regex patterns and Windows paths (or use forward slashes and `pathlib`).
- Triple quotes span lines — ideal for prompts — but the indentation of the code leaks into the text. `textwrap.dedent` removes the common leading whitespace.
- Adjacent string literals are joined at compile time — handy for long lines, and a silent bug when you forget a comma in a list.

**Predict, then run:** how many stop sequences are in `stop`?

In [ ]:
import textwrap

path = "C:\new\table"
print("len(path) =", len(path), "->", repr(path))
print("raw       :", r"C:\new\table", "| regex:", r"\d+\.\d+")

def build_prompt(question):
    prompt = """
        You are a support assistant.
        Answer in one short paragraph.
        Question: {q}
        """
    return textwrap.dedent(prompt).strip().format(q=question)

print(build_prompt("How do refunds work?"))

stop = ["\n\n", "User:" "Assistant:"]            # missing comma!
print("stop =", stop, "| len:", len(stop))
assert len(path) == 10 and build_prompt("x").startswith("You are") and len(stop) == 2

## 6 · `str` vs `bytes`, and UTF-8

**In plain English:** a `str` is a sequence of Unicode **code points** (abstract characters). Files, sockets and HTTP carry **bytes**. UTF-8 is the encoding nearly everything uses to turn one into the other: ASCII takes 1 byte per character, accented Latin letters 2, most Asian scripts 3, emoji 4. `encode()` goes str → bytes, `decode()` goes bytes → str.

So "how long is this text?" has at least four answers: code points (`len(s)`), UTF-8 bytes (`len(s.encode())`), what a human sees as characters (grapheme clusters — `👍🏽` is 2 code points, and `é` can be 1 or 2), and LLM **tokens**, which depend on the tokenizer ([tokenization](../07_genai_foundations/01_tokenization_bpe_and_tiktoken.ipynb)).

In [ ]:
samples = ["a", "é", "€", "你", "😀", "👍🏽", "e\u0301"]
print(f"{'text':<6}{'len(str)':>9}{'utf-8 bytes':>13}   code points")
for ch in samples:
    points = " ".join(f"U+{ord(c):04X}" for c in ch)
    print(f"{ch:<6}{len(ch):>9}{len(ch.encode('utf-8')):>13}   {points}")

import sys                   # in memory, CPython stores 1, 2 or 4 bytes per character — set by the widest one
ascii_only, with_emoji = "a" * 100, "a" * 99 + "😀"
print(f"\nin memory: 100 ASCII chars = {sys.getsizeof(ascii_only)} bytes; 99 ASCII + 1 emoji = {sys.getsizeof(with_emoji)} bytes")
assert sys.getsizeof(with_emoji) > 3 * sys.getsizeof(ascii_only)
assert [len(c.encode()) for c in samples[:5]] == [1, 2, 3, 3, 4] and len("👍🏽") == 2

In [ ]:
import matplotlib.pyplot as plt

greetings = {"English": "Good morning", "French": "Bonjour à tous", "Hindi": "सुप्रभात", "Chinese": "早上好", "Emoji": "☀️🙂👋"}
chars = [len(t) for t in greetings.values()]
nbytes = [len(t.encode("utf-8")) for t in greetings.values()]
x = list(range(len(greetings)))
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar([i - 0.2 for i in x], chars, 0.4, label="len(text): code points")
ax.bar([i + 0.2 for i in x], nbytes, 0.4, label="len(text.encode('utf-8')): bytes")
ax.set_xticks(x, list(greetings))
ax.set_title("A greeting in five scripts: characters vs UTF-8 bytes")
ax.set_ylabel("count"); ax.legend()
plt.show()
print({lang: f"{c} code points, {b} bytes" for lang, c, b in zip(greetings, chars, nbytes)})

The emoji greeting shows **4** code points for 3 visible symbols: `☀️` is the sun (U+2600) plus an invisible *variation selector* (U+FE0F) asking for the colour-emoji style. Byte limits (API payloads, database columns) and character limits are different budgets — and neither is the token budget.

### Encoding bugs you will actually meet

- **Mojibake:** decoding UTF-8 bytes with the wrong codec (`latin-1`, `cp1252`) turns `café` into `cafÃ©`. Always pass `encoding="utf-8"` when opening files.
- **Look-alike strings:** `é` can be one code point (U+00E9) or `e` + a combining accent (U+0301). They print the same but compare unequal — normalise with `unicodedata.normalize("NFC", s)` before comparing, deduplicating or keyword-matching.
- **Cutting bytes mid-character:** truncating the *encoded* bytes can split a multi-byte character, and `decode()` fails. Truncate the `str`, or decode with `errors="ignore"`. When bytes arrive in chunks — a streaming HTTP response — use an **incremental decoder**, which holds back an incomplete character until the rest arrives.

In [ ]:
import codecs
import unicodedata

raw = "café".encode("utf-8")
print("utf-8 bytes     :", raw)
print("wrong codec     :", raw.decode("latin-1"))
print("right codec     :", raw.decode("utf-8"))

composed, decomposed = "caf\u00e9", "cafe\u0301"
print("look alike, equal?", composed == decomposed, "| equal after NFC?", unicodedata.normalize("NFC", decomposed) == composed)

emoji_bytes = "ok 😀".encode("utf-8")               # 3 ASCII bytes + 4 emoji bytes
try:
    emoji_bytes[:5].decode("utf-8")
except UnicodeDecodeError as err:
    print("cut mid-char    : UnicodeDecodeError —", err.reason)
print("errors='ignore' :", repr(emoji_bytes[:5].decode("utf-8", errors="ignore")))

decoder = codecs.getincrementaldecoder("utf-8")()
network_chunks = [emoji_bytes[:5], emoji_bytes[5:]]  # the emoji is split across two chunks
pieces = [decoder.decode(chunk) for chunk in network_chunks]
print("incremental     :", pieces)
assert "".join(pieces) == "ok 😀" and pieces[0] == "ok "

## 7 · Building strings: `"".join` vs `+=`

Strings are immutable, so `s += piece` must create a new string and copy everything built so far — **O(n²)** total for n pieces. CPython has a special case that can resize the string in place instead, but it only applies to a plain local variable inside a function with no other references to the string, and it is an implementation detail other Pythons do not promise. `"".join(pieces)` computes the final size once and copies each piece once — **O(n)**, everywhere.

**Predict, then run:** which of the four lines grows fastest as `n` doubles?

In [ ]:
import timeit

def build_join(pieces):
    return "".join(pieces)

def build_plus_local(pieces):            # CPython's in-place trick can apply here
    s = ""
    for p in pieces:
        s += p
    return s

def build_plus_dict(pieces):             # building into a dict value: the trick cannot apply
    doc = {"text": ""}
    for p in pieces:
        doc["text"] += p
    return doc["text"]

def build_plus_shared(pieces):           # a second name refers to the string: the trick cannot apply
    s = ""
    for p in pieces:
        previous = s
        s += p
    return s

sizes = [1_000, 2_000, 4_000, 8_000]
all_pieces = [f"token_{i:05d} " for i in range(max(sizes))]
slices = {n: all_pieces[:n] for n in sizes}
funcs = [build_join, build_plus_local, build_plus_dict, build_plus_shared]
ms = {f.__name__: [min(timeit.repeat(lambda: f(slices[n]), number=1, repeat=3)) * 1000 for n in sizes] for f in funcs}
for name, times in ms.items():
    print(f"{name:<18}", "  ".join(f"{t:7.2f} ms" for t in times))
assert all(f(slices[1_000]) == "".join(slices[1_000]) for f in funcs)
assert ms["build_plus_dict"][-1] > 5 * ms["build_join"][-1] and ms["build_plus_shared"][-1] > 5 * ms["build_join"][-1]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for name, times in ms.items():
    ax.plot(sizes, times, marker="o", label=name)
ax.set_title("Building one string from n pieces")
ax.set_xlabel("number of pieces n"); ax.set_ylabel("time (ms)")
ax.legend()
plt.show()

The two "trick cannot apply" lines grow by roughly 3–4× each time `n` doubles — the signature of O(n²) — while `join` stays near zero and the optimised local `+=` grows linearly but is still several times slower than `join`. Code at the **top level of a notebook cell** runs at module scope, where CPython 3.12's in-place trick does not apply either, so a `+=` loop written directly in a cell behaves like the slow lines. Write `"".join(...)` (or collect pieces in a list, or use `io.StringIO`) and the question never arises.

## 8 · Practical: clean text before it goes to an LLM

Text scraped from web pages, PDFs and tickets arrives with junk that wastes tokens and hurts retrieval: non-breaking spaces, zero-width characters, ligatures (`ﬁ`), HTML entities, runs of blank lines, trailing spaces. A small, boring cleaning function pays for itself in every RAG pipeline:

1. `html.unescape` — `&amp;` → `&`.
2. `unicodedata.normalize("NFKC", …)` — folds compatibility characters: non-breaking space → space, `ﬁ` → `fi`, full-width `Ａ` → `A`.
3. Drop invisible format characters (Unicode category `Cf`, such as the zero-width space U+200B).
4. Collapse whitespace inside each paragraph but keep paragraph breaks.
5. Optionally truncate to a character budget at a word boundary.

In [ ]:
import html

def clean_text(raw: str, max_chars: int | None = None) -> str:
    text = html.unescape(raw)
    text = unicodedata.normalize("NFKC", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Cf")
    paragraphs = [" ".join(p.split()) for p in text.split("\n\n")]
    text = "\n\n".join(p for p in paragraphs if p)
    if max_chars is not None and len(text) > max_chars:
        budget = max_chars - 2                                   # leave room for " …"
        boundary = max(text.rfind(" ", 0, budget + 1), text.rfind("\n", 0, budget + 1))
        text = (text[:boundary] if boundary > 0 else text[:budget]).rstrip() + " …"
    return text

raw = "Refund\u00a0policy &amp; exceptions\u200b\n\n\n   Customers   can \ufb01le a claim \n within 30 days.  \n\n\n"
cleaned = clean_text(raw)
print("before:", repr(raw))
print("after :", repr(cleaned))
print(f"chars : {len(raw)} -> {len(cleaned)} | truncated to 30: {clean_text(raw, 30)!r}")
assert "\u00a0" not in cleaned and "\u200b" not in cleaned and "  " not in cleaned and "file a claim" in cleaned
assert clean_text(raw, 30) == "Refund policy & exceptions …"      # cut at a word boundary, within budget
assert all(len(clean_text(raw, n)) <= n for n in range(10, 80))

Cleaned text then goes into a prompt inside clear delimiters, so the model can tell your instructions from the document (more in [prompt engineering](../08_llm_apps/02_prompt_engineering.ipynb)):

In [ ]:
question = "How long do customers have to file a claim?"
prompt = f"""Answer using only the context. If the answer is not there, say you don't know.

<context>
{clean_text(raw, max_chars=200)}
</context>

Question: {question}"""
print(prompt)

## Try it yourself

**1.** Write `is_palindrome(text)` that ignores case, spaces and punctuation. Test it on `"A man, a plan, a canal: Panama"`.

In [ ]:
# Solution — try it yourself first, then run this
def is_palindrome(text: str) -> bool:
    letters = [ch.casefold() for ch in text if ch.isalnum()]
    return letters == letters[::-1]

assert is_palindrome("A man, a plan, a canal: Panama")
assert is_palindrome("No 'x' in Nixon")
assert not is_palindrome("RAG is great")
print("is_palindrome passes")

**2.** Write `parse_action(line)` that turns `"Action: calculator[17 * 23]"` into `("calculator", "17 * 23")` and returns `None` for any line that is not an action.

In [ ]:
# Solution — try it yourself first, then run this
def parse_action(line: str):
    key, sep, rest = line.partition(": ")
    if key != "Action" or not sep or not rest.endswith("]") or "[" not in rest:
        return None
    tool, _, arg = rest.removesuffix("]").partition("[")
    return tool.strip(), arg

print(parse_action("Action: calculator[17 * 23]"))
assert parse_action("Action: calculator[17 * 23]") == ("calculator", "17 * 23")
assert parse_action("Thought: hmm") is None and parse_action("Action: broken") is None

**3.** Print a token-usage report: one row per step, the step name left-aligned in 12 characters, tokens right-aligned with a thousands separator, and each step's share of the total as a percentage with one decimal.

In [ ]:
# Solution — try it yourself first, then run this
usage = {"retrieve": 0, "rerank": 1_250, "generate": 18_400, "judge": 3_100}
total = sum(usage.values())
lines = [f"{step:<12}{tokens:>9,}{tokens / total:>8.1%}" for step, tokens in usage.items()]
print("\n".join(lines))
print(f"{'total':<12}{total:>9,}{1:>8.1%}")
assert lines[2] == "generate       18,400   80.9%" and all(len(line) == 12 + 9 + 8 for line in lines)

**4.** An API limits a field to `max_bytes` of UTF-8. Write `truncate_utf8(text, max_bytes)` that never splits a character (and never raises).

In [ ]:
# Solution — try it yourself first, then run this
def truncate_utf8(text: str, max_bytes: int) -> str:
    return text.encode("utf-8")[:max_bytes].decode("utf-8", errors="ignore")

for limit in range(0, 9):
    out = truncate_utf8("ok 😀!", limit)
    assert len(out.encode("utf-8")) <= limit and "ok 😀!".startswith(out)
print(truncate_utf8("ok 😀!", 5), "|", truncate_utf8("ok 😀!", 7))

## Say it in an interview

- **Strings (30 s):** "A `str` is an immutable sequence of Unicode code points. Slicing is `[start:stop:step]`, `stop` exclusive, and never raises — `s[::-1]` reverses. Every method returns a new string."
- **Building strings:** "`''.join(parts)` is O(n); `+=` in a loop is O(n²) in general. CPython sometimes optimises `+=` in place, but that is fragile — it doesn't apply at module level, to attributes or dict values, or when another reference exists."
- **Unicode:** "`encode` goes str → bytes, `decode` bytes → str; UTF-8 is variable-width, 1–4 bytes per code point. `len()` counts code points — not bytes, not visible characters, not tokens. Normalise (NFC/NFKC) and `casefold()` before comparing or deduplicating text."
- **Formatting:** `f"{x:.2f}"`, `f"{n:,}"`, `f"{r:.1%}"`, `f"{s:>10}"`, `f"{x!r}"`, and `f"{x=}"` for debugging. `str.format` when the template is data; `string.Template` when the template contains JSON braces.
- **Traps:** `strip()` takes a *set of characters* (use `removeprefix`); `split()` vs `split(" ")`; `.format()` on a template with JSON examples raises `KeyError`; a missing comma silently concatenates two list items; `"C:\new"` contains a newline; `title()` mangles apostrophes.
- **Follow-up — "how do you count tokens?"** With the model's own tokenizer (e.g. `tiktoken` for OpenAI models); roughly 4 characters per token for English, and noticeably more tokens per character for many other scripts.

## Next

- [03 · Lists and tuples](03_lists_and_tuples.ipynb) — sequences you *can* change.
- [Tokenization, BPE and tiktoken](../07_genai_foundations/01_tokenization_bpe_and_tiktoken.ipynb) — how text becomes tokens.
- [Chunking strategies](../09_rag/01_chunking_strategies_and_semantic_chunking.ipynb) — what happens to cleaned text next.
- [String problems](../02_python_problems/02_string_problems.ipynb) — interview-style practice.
- Theory: the [python_basics course](../../python_basics/index.html) (chapter 3: strings and slicing) · the [interview bank](../../ai_interview_prep/index.html).